# CSV 정규화

flaws_cloudtrail19 를 기준으로 작성했습니다.   
해당 로그를 통해 4개의 csv 파일이 추출됩니다.  

- `identities.csv`
- `events.csv`
- `resources.csv`
- `relationships.csv`

In [ ]:
import json
import re
import hashlib
import pandas as pd
from collections import defaultdict

from pathlib import Path

BASE_DIR = Path("..")

INPUT_PATH = BASE_DIR / "data" / "raw" / "flaws_cloudtrail_logs" / "flaws_cloudtrail19.json"
OUTPUT_DIR = BASE_DIR / "data" / "normalized"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with INPUT_PATH.open("r", encoding="utf-8") as f:
    raw = json.load(f)

records = raw.get("Records", [])
print(f"Loaded {len(records):,} CloudTrail records from {INPUT_PATH}")

IDENTITY_TYPE_MAP = {
    "Root": ("ROOT_USER", "STATIC"),
    "IAMUser": ("IAM_USER", "STATIC"),
    "AssumedRole": ("ROLE", "SESSION"),
    "FederatedUser": ("FEDERATED_IDENTITY", "SESSION"),
    "AWSService": ("SERVICE_PRINCIPAL", "STATIC"),
    "AWSAccount": ("UNKNOWN_IDENTITY", "UNKNOWN"),
}

READ_PREFIXES = (
    "Get", "List", "Describe", "Lookup", "Search", "Head",
    "BatchGet", "Select", "Scan", "Query"
)
CREATE_PREFIXES = ("Create", "Register", "Import", "Allocate", "Launch")
DELETE_PREFIXES = ("Delete", "Terminate", "Remove", "Deregister", "Deallocate")
WRITE_PREFIXES = ("Put", "Write", "Upload", "BatchWrite")
EXECUTE_PREFIXES = ("Invoke", "Run", "Start", "Execute", "SendCommand")
MODIFY_PREFIXES = (
    "Update", "Modify", "Set", "Attach", "Detach", "Enable", "Disable",
    "Authorize", "Revoke", "Tag", "Untag", "Associate", "Disassociate"
)

EVENT_OVERRIDES = {
    "AssumeRole": "ASSUME",
    "AssumeRoleWithSAML": "ASSUME",
    "AssumeRoleWithWebIdentity": "ASSUME",
    "ConsoleLogin": "AUTHENTICATE",
}

RESOURCE_TYPE_BY_CFN = {
    "AWS::S3::Bucket": "STORAGE",
    "AWS::S3::Object": "STORAGE",
    "AWS::Lambda::Function": "COMPUTE",
    "AWS::EC2::Instance": "COMPUTE",
    "AWS::ECS::Task": "COMPUTE",
    "AWS::EKS::Cluster": "COMPUTE",
    "AWS::DynamoDB::Table": "DATABASE",
    "AWS::RDS::DBInstance": "DATABASE",
    "AWS::Redshift::Cluster": "DATABASE",
    "AWS::SecretsManager::Secret": "SECRET",
    "AWS::KMS::Key": "KEY",
    "AWS::EC2::SecurityGroup": "SECURITY_CONTROL",
    "AWS::SQS::Queue": "QUEUE",
}

SERVICE_RESOURCE_TYPE = {
    "s3": "STORAGE",
    "lambda": "COMPUTE",
    "ec2": "COMPUTE",
    "ecs": "COMPUTE",
    "eks": "COMPUTE",
    "sagemaker": "COMPUTE",
    "dynamodb": "DATABASE",
    "rds": "DATABASE",
    "redshift": "DATABASE",
    "elasticache": "DATABASE",
    "secretsmanager": "SECRET",
    "kms": "KEY",
    "cloudhsm": "KEY",
    "sqs": "QUEUE",
    "sns": "QUEUE",
    "apigateway": "NETWORK",
    "cloudfront": "NETWORK",
    "waf": "SECURITY_CONTROL",
}

def service_name(event_source):
    if not event_source:
        return ""
    return str(event_source).split(".")[0].lower()

def semantic_event_type(record):
    name = str(record.get("eventName") or "")
    if name in EVENT_OVERRIDES:
        return EVENT_OVERRIDES[name]
    if record.get("readOnly") is True:
        return "READ"
    if name.startswith(READ_PREFIXES):
        return "READ"
    if name.startswith(DELETE_PREFIXES):
        return "DELETE"
    if name.startswith(CREATE_PREFIXES):
        return "CREATE"
    if name.startswith(EXECUTE_PREFIXES):
        return "EXECUTE"
    if name.startswith(WRITE_PREFIXES):
        return "WRITE"
    if name.startswith(MODIFY_PREFIXES):
        return "MODIFY"
    # CloudTrail에 readOnly가 없고 prefix rule에도 걸리지 않는 경우
    # 원본 eventName은 보존하고, 의미 분류는 MODIFY로 보수적으로 둡니다.
    return "MODIFY"

def outcome(record):
    return "FAILURE" if record.get("errorCode") else "SUCCESS"

def stable_hash(*parts, prefix="id"):
    raw = "|".join("" if p is None else str(p) for p in parts)
    return f"{prefix}:{hashlib.sha256(raw.encode()).hexdigest()[:20]}"

def name_from_arn(arn):
    if not arn:
        return ""
    tail = str(arn).split(":", 5)[-1]
    return tail.rstrip("/").split("/")[-1]

def arn_service(arn):
    if not arn or not str(arn).startswith("arn:"):
        return ""
    parts = str(arn).split(":", 5)
    return parts[2] if len(parts) > 2 else ""

def arn_account(arn):
    if not arn or not str(arn).startswith("arn:"):
        return ""
    parts = str(arn).split(":", 5)
    return parts[4] if len(parts) > 4 else ""

def identity_id_from_arn_or_fallback(arn=None, account_id=None, principal=None, raw_type=None):
    if arn:
        return f"aws:identity:{arn}"
    return stable_hash("AWS", account_id, principal, raw_type, prefix="aws:identity")

def identity_from_user_identity(record):
    ui = record.get("userIdentity") or {}
    raw_type = ui.get("type") or "Unknown"
    identity_type, identity_state = IDENTITY_TYPE_MAP.get(
        raw_type, ("UNKNOWN_IDENTITY", "UNKNOWN")
    )

    arn = ui.get("arn") or ""
    account_id = ui.get("accountId") or record.get("recipientAccountId") or ""

    if raw_type == "AWSService":
        name = ui.get("invokedBy") or ui.get("principalId") or "AWSService"
        principal = name
    else:
        name = (
            ui.get("userName")
            or name_from_arn(arn)
            or ui.get("principalId")
            or account_id
            or "unknown"
        )
        principal = ui.get("principalId") or name

    return {
        "identity_id": identity_id_from_arn_or_fallback(
            arn, account_id, principal, raw_type
        ),
        "category": "IDENTITY",
        "identity_type": identity_type,
        "identity_state": identity_state,
        "name": name,
        "arn": arn,
        "account_id": account_id,
        "provider": "AWS",
        "raw_identity_type": raw_type,
    }

def session_issuer_identity(record):
    ui = record.get("userIdentity") or {}
    issuer = ((ui.get("sessionContext") or {}).get("sessionIssuer") or {})
    if not issuer:
        return None

    raw_type = issuer.get("type") or "Role"
    arn = issuer.get("arn") or ""
    account_id = issuer.get("accountId") or ""
    name = issuer.get("userName") or name_from_arn(arn) or issuer.get("principalId") or "unknown"

    identity_type = "ROLE" if raw_type == "Role" else IDENTITY_TYPE_MAP.get(
        raw_type, ("UNKNOWN_IDENTITY", "STATIC")
    )[0]

    return {
        "identity_id": identity_id_from_arn_or_fallback(
            arn, account_id, issuer.get("principalId"), raw_type
        ),
        "category": "IDENTITY",
        "identity_type": identity_type,
        "identity_state": "STATIC",
        "name": name,
        "arn": arn,
        "account_id": account_id,
        "provider": "AWS",
        "raw_identity_type": raw_type,
    }

def identity_target_from_arn(arn, raw_type=None, account_id=None):
    if not arn:
        return None

    arn = str(arn)
    resource_part = arn.split(":", 5)[-1] if arn.startswith("arn:") else ""

    if ":iam:" in arn and (resource_part.startswith("role/") or raw_type == "AWS::IAM::Role"):
        identity_type = "ROLE"
        raw_identity_type = raw_type or "AWS::IAM::Role"
    elif ":iam:" in arn and (resource_part.startswith("user/") or raw_type == "AWS::IAM::User"):
        identity_type = "IAM_USER"
        raw_identity_type = raw_type or "AWS::IAM::User"
    elif ":iam:" in arn and resource_part == "root":
        identity_type = "ROOT_USER"
        raw_identity_type = raw_type or "AWS::IAM::Root"
    else:
        return None

    account_id = account_id or arn_account(arn)
    return {
        "identity_id": identity_id_from_arn_or_fallback(
            arn, account_id, name_from_arn(arn), raw_identity_type
        ),
        "category": "IDENTITY",
        "identity_type": identity_type,
        "identity_state": "STATIC",
        "name": name_from_arn(arn),
        "arn": arn,
        "account_id": account_id,
        "provider": "AWS",
        "raw_identity_type": raw_identity_type,
    }

def resource_type(raw_type=None, service=None):
    if raw_type in RESOURCE_TYPE_BY_CFN:
        return RESOURCE_TYPE_BY_CFN[raw_type]
    return SERVICE_RESOURCE_TYPE.get(service or "", "UNKNOWN_RESOURCE")

def resource_id(arn=None, service=None, account_id=None, region=None, name=None, raw_type=None):
    if arn:
        return f"aws:resource:{arn}"
    return stable_hash(
        "AWS", service, account_id, region, name, raw_type,
        prefix="aws:resource"
    )

def make_resource(*, arn="", raw_type="", name="", service="", account_id="", region=""):
    return {
        "resource_id": resource_id(
            arn=arn, service=service, account_id=account_id,
            region=region, name=name, raw_type=raw_type
        ),
        "category": "RESOURCE",
        "resource_type": resource_type(raw_type, service),
        "raw_resource_type": raw_type or "",
        "name": name or name_from_arn(arn),
        "arn": arn or "",
        "service": service or arn_service(arn),
        "account_id": account_id or arn_account(arn),
        "region": region or "",
    }

def target_from_arn(arn, raw_type=None, account_id=None, region=None,
                    evidence_type="DIRECT_CLOUDTRAIL", confidence=1.0):
    ident = identity_target_from_arn(arn, raw_type, account_id)
    if ident:
        return {
            "category": "IDENTITY",
            "node": ident,
            "evidence_type": evidence_type,
            "confidence": confidence,
        }

    service = arn_service(arn)
    res = make_resource(
        arn=arn,
        raw_type=raw_type or "",
        service=service,
        account_id=account_id or arn_account(arn),
        region=region or "",
    )
    return {
        "category": "RESOURCE",
        "node": res,
        "evidence_type": evidence_type,
        "confidence": confidence,
    }

def request_parameter_targets(record):
    """대표 서비스의 requestParameters에서 실제 target을 복원합니다.
    모르는 구조는 억지로 생성하지 않고 건너뜁니다.
    """
    params = record.get("requestParameters")
    if not isinstance(params, dict):
        return []

    svc = service_name(record.get("eventSource"))
    region = record.get("awsRegion") or ""
    account_id = record.get("recipientAccountId") or (record.get("userIdentity") or {}).get("accountId") or ""
    targets = []

    def add_resource(name, raw_type, arn=""):
        if not name:
            return
        targets.append({
            "category": "RESOURCE",
            "node": make_resource(
                arn=arn,
                raw_type=raw_type,
                name=str(name),
                service=svc,
                account_id=account_id,
                region=region,
            ),
            "evidence_type": "RULE_DERIVED",
            "confidence": 0.95,
        })

    # S3
    if svc == "s3":
        bucket = params.get("bucketName")
        key = params.get("key")
        if bucket and key:
            add_resource(
                f"{bucket}/{key}",
                "AWS::S3::Object",
                f"arn:aws:s3:::{bucket}/{key}"
            )
        elif bucket:
            add_resource(
                bucket,
                "AWS::S3::Bucket",
                f"arn:aws:s3:::{bucket}"
            )

    # Lambda
    elif svc == "lambda":
        fn = params.get("functionName")
        if fn:
            arn = fn if str(fn).startswith("arn:") else ""
            add_resource(fn, "AWS::Lambda::Function", arn)

    # Secrets Manager
    elif svc == "secretsmanager":
        secret = params.get("secretId")
        if secret:
            arn = secret if str(secret).startswith("arn:") else ""
            add_resource(secret, "AWS::SecretsManager::Secret", arn)

    # KMS
    elif svc == "kms":
        key_id = params.get("keyId")
        if key_id:
            arn = key_id if str(key_id).startswith("arn:") else ""
            add_resource(key_id, "AWS::KMS::Key", arn)

    # DynamoDB
    elif svc == "dynamodb":
        table = params.get("tableName")
        if table:
            add_resource(table, "AWS::DynamoDB::Table")

    # RDS
    elif svc == "rds":
        db = params.get("dBInstanceIdentifier") or params.get("dbInstanceIdentifier")
        if db:
            add_resource(db, "AWS::RDS::DBInstance")

    # EC2 - instanceIds / instancesSet.items[].instanceId
    elif svc == "ec2":
        ids = []
        if params.get("instanceId"):
            ids.append(params["instanceId"])
        if isinstance(params.get("instanceIds"), list):
            ids.extend(params["instanceIds"])
        items = ((params.get("instancesSet") or {}).get("items") or [])
        for item in items:
            if isinstance(item, dict) and item.get("instanceId"):
                ids.append(item["instanceId"])
        for instance_id in dict.fromkeys(map(str, ids)):
            add_resource(instance_id, "AWS::EC2::Instance")

    return targets

def extract_targets(record):
    """Event의 target들을 추출합니다.
    우선순위:
      1) CloudTrail resources[] 직접 관측
      2) requestParameters 기반 resolver
    동일 target은 direct evidence를 우선하여 중복 제거합니다.
    """
    found = {}

    for raw_res in record.get("resources") or []:
        if not isinstance(raw_res, dict):
            continue
        arn = raw_res.get("ARN") or raw_res.get("arn")
        raw_type = raw_res.get("type") or ""
        if not arn:
            continue

        target = target_from_arn(
            arn=arn,
            raw_type=raw_type,
            account_id=raw_res.get("accountId"),
            region=record.get("awsRegion"),
            evidence_type="DIRECT_CLOUDTRAIL",
            confidence=1.0,
        )
        node_id = (
            target["node"]["identity_id"]
            if target["category"] == "IDENTITY"
            else target["node"]["resource_id"]
        )
        found[(target["category"], node_id)] = target

    for target in request_parameter_targets(record):
        node_id = (
            target["node"]["identity_id"]
            if target["category"] == "IDENTITY"
            else target["node"]["resource_id"]
        )
        key = (target["category"], node_id)
        # direct CloudTrail evidence가 있으면 덮어쓰지 않음
        found.setdefault(key, target)

    return list(found.values())

def upsert_seen(store, node, event_time):
    """동일 Identity/Resource를 중복 생성하지 않고 first_seen/last_seen을 갱신합니다."""
    key = node.get("identity_id") or node.get("resource_id")
    if key not in store:
        store[key] = dict(node)
        store[key]["first_seen"] = event_time
        store[key]["last_seen"] = event_time
    else:
        if event_time:
            first = store[key].get("first_seen")
            last = store[key].get("last_seen")
            if not first or event_time < first:
                store[key]["first_seen"] = event_time
            if not last or event_time > last:
                store[key]["last_seen"] = event_time

print("Normalization helpers ready.")

FileNotFoundError: [Errno 2] No such file or directory: '\\mnt\\data\\flaws_cloudtrail19.json'

## 1. `identities.csv`

In [ ]:
# ============================================================
# BLOCK 1 — identities.csv
# ============================================================

identity_store = {}

for record in records:
    event_time = record.get("eventTime") or ""

    # 1) 실제 API 호출자
    actor = identity_from_user_identity(record)
    upsert_seen(identity_store, actor, event_time)

    # 2) AssumedRole인 경우 sessionIssuer(원래 IAM Role)도 별도 STATIC Identity로 보존
    issuer = session_issuer_identity(record)
    if issuer:
        upsert_seen(identity_store, issuer, event_time)

    # 3) AssumeRole 등의 target이 IAM Role/User인 경우 RESOURCE가 아닌 IDENTITY로 보존
    for target in extract_targets(record):
        if target["category"] == "IDENTITY":
            upsert_seen(identity_store, target["node"], event_time)

identity_columns = [
    "identity_id",
    "category",
    "identity_type",
    "identity_state",
    "name",
    "arn",
    "account_id",
    "provider",
    "raw_identity_type",
    "first_seen",
    "last_seen",
]

identities_df = pd.DataFrame(identity_store.values())
if identities_df.empty:
    identities_df = pd.DataFrame(columns=identity_columns)
else:
    identities_df = identities_df[identity_columns].sort_values(
        ["identity_type", "name", "identity_id"]
    ).reset_index(drop=True)

identities_path = OUTPUT_DIR / "identities.csv"
identities_df.to_csv(identities_path, index=False)

print(f"identities.csv: {len(identities_df):,} rows -> {identities_path}")
display(identities_df.head(10))

identities.csv: 62 rows -> normalized_csv/identities.csv


,identity_id,category,identity_type,identity_state,name,arn,account_id,provider,raw_identity_type,first_seen,last_seen
0,aws:identity:arn:aws:iam::811596193553:user/Le...,IDENTITY,IAM_USER,STATIC,Level6,arn:aws:iam::811596193553:user/Level6,811596193553,AWS,IAMUser,2020-09-21T22:22:52Z,2020-10-07T14:52:39Z
1,aws:identity:arn:aws:iam::811596193553:user/ba...,IDENTITY,IAM_USER,STATIC,backup,arn:aws:iam::811596193553:user/backup,811596193553,AWS,IAMUser,2020-09-22T02:20:02Z,2020-10-07T21:03:30Z
2,aws:identity:arn:aws:iam::811596193553:role/aw...,IDENTITY,ROLE,STATIC,AWSServiceRoleForConfigMultiAccountSetup,arn:aws:iam::811596193553:role/aws-service-rol...,811596193553,AWS,AWS::IAM::Role,2020-09-22T08:38:55Z,2020-10-07T08:39:13Z
3,aws:identity:arn:aws:iam::811596193553:role/se...,IDENTITY,ROLE,STATIC,Level6,arn:aws:iam::811596193553:role/service-role/Le...,811596193553,AWS,AWS::IAM::Role,2020-09-22T14:32:30Z,2020-10-07T14:53:06Z
4,aws:identity:arn:aws:sts::811596193553:assumed...,IDENTITY,ROLE,SESSION,Level6,arn:aws:sts::811596193553:assumed-role/Level6/...,811596193553,AWS,AssumedRole,2020-09-22T14:32:39Z,2020-10-07T14:53:06Z
5,aws:identity:arn:aws:iam::811596193553:role/aw...,IDENTITY,ROLE,STATIC,aws:ec2-instance,arn:aws:iam::811596193553:role/aws:ec2-instance,811596193553,AWS,Role,2020-09-24T14:19:51Z,2020-10-06T16:57:12Z
6,aws:identity:arn:aws:sts::811596193553:assumed...,IDENTITY,ROLE,SESSION,botocore-session-aed6ad4c04,arn:aws:sts::811596193553:assumed-role/tests3a...,811596193553,AWS,AssumedRole,2020-10-07T20:27:01Z,2020-10-07T20:27:01Z
7,aws:identity:arn:aws:iam::811596193553:role/se...,IDENTITY,ROLE,STATIC,config-role-us-west-2,arn:aws:iam::811596193553:role/service-role/co...,811596193553,AWS,AWS::IAM::Role,2020-09-22T02:10:54Z,2020-10-07T20:17:38Z
8,aws:identity:arn:aws:iam::811596193553:role/flaws,IDENTITY,ROLE,STATIC,flaws,arn:aws:iam::811596193553:role/flaws,811596193553,AWS,AWS::IAM::Role,2020-09-21T23:02:39Z,2020-10-07T20:57:09Z
9,aws:identity:arn:aws:sts::811596193553:assumed...,IDENTITY,ROLE,SESSION,i-aa2d3b42e5c6e801a,arn:aws:sts::811596193553:assumed-role/aws:ec2...,811596193553,AWS,AssumedRole,2020-09-24T14:19:51Z,2020-10-06T16:57:12Z


## 2. `events.csv`

In [ ]:
# ============================================================
# BLOCK 2 — events.csv
# ============================================================

event_rows = []

for index, record in enumerate(records):
    event_id = record.get("eventID") or stable_hash(
        record.get("eventTime"),
        record.get("eventSource"),
        record.get("eventName"),
        record.get("requestID"),
        index,
        prefix="aws:event",
    )

    event_rows.append({
        "event_id": event_id,
        "category": "EVENT",
        "event_type": semantic_event_type(record),
        "event_source": record.get("eventSource") or "",
        "raw_event_name": record.get("eventName") or "",
        "event_time": record.get("eventTime") or "",
        "region": record.get("awsRegion") or "",
        "source_ip": record.get("sourceIPAddress") or "",
        "user_agent": record.get("userAgent") or "",
        "outcome": outcome(record),
        "raw_log_ref": f"{INPUT_PATH.name}#eventID={event_id}",
    })

event_columns = [
    "event_id",
    "category",
    "event_type",
    "event_source",
    "raw_event_name",
    "event_time",
    "region",
    "source_ip",
    "user_agent",
    "outcome",
    "raw_log_ref",
]

events_df = pd.DataFrame(event_rows, columns=event_columns)
events_path = OUTPUT_DIR / "events.csv"
events_df.to_csv(events_path, index=False)

print(f"events.csv: {len(events_df):,} rows -> {events_path}")
display(events_df.head(10))

events.csv: 39,207 rows -> normalized_csv/events.csv


,event_id,category,event_type,event_source,raw_event_name,event_time,region,source_ip,user_agent,outcome,raw_log_ref
0,f146e1b4-5796-40b0-aa15-ba54d7d823d5,EVENT,READ,cloudfront.amazonaws.com,ListCloudFrontOriginAccessIdentities,2020-09-21T22:22:52Z,us-east-1,205.8.181.128,Boto3/1.15.2 Python/3.8.2 Linux/5.6.3-arch1-1 ...,SUCCESS,flaws_cloudtrail19.json#eventID=f146e1b4-5796-...
1,8c58977-02ee-4dc3-8784-7f4bbe90c591,EVENT,READ,cloudfront.amazonaws.com,ListStreamingDistributions,2020-09-21T22:22:52Z,us-east-1,205.8.181.128,Boto3/1.15.2 Python/3.8.2 Linux/5.6.3-arch1-1 ...,SUCCESS,flaws_cloudtrail19.json#eventID=8c58977-02ee-4...
2,e2f01b9a-c3d5-4429-9a98-bc7774fb0c41,EVENT,READ,cloudfront.amazonaws.com,ListDistributions,2020-09-21T22:22:52Z,us-east-1,205.8.181.128,Boto3/1.15.2 Python/3.8.2 Linux/5.6.3-arch1-1 ...,SUCCESS,flaws_cloudtrail19.json#eventID=e2f01b9a-c3d5-...
3,64220d-0873-4ffb-bb1d-7b9d94c5c773,EVENT,READ,cloudfront.amazonaws.com,ListStreamingDistributions,2020-09-21T22:22:52Z,us-east-1,205.8.181.128,Boto3/1.15.2 Python/3.8.2 Linux/5.6.3-arch1-1 ...,SUCCESS,flaws_cloudtrail19.json#eventID=64220d-0873-4f...
4,d35b87ef-fcbf-4091-802f-a6eafdce7132,EVENT,READ,cloudfront.amazonaws.com,ListStreamingDistributions,2020-09-21T22:22:52Z,us-east-1,205.8.181.128,Boto3/1.15.2 Python/3.8.2 Linux/5.6.3-arch1-1 ...,SUCCESS,flaws_cloudtrail19.json#eventID=d35b87ef-fcbf-...
5,0f24b61e-747e-42f0-ab44-bd9f3cf08c1b,EVENT,READ,cloudfront.amazonaws.com,ListCloudFrontOriginAccessIdentities,2020-09-21T22:22:52Z,us-east-1,205.8.181.128,Boto3/1.15.2 Python/3.8.2 Linux/5.6.3-arch1-1 ...,SUCCESS,flaws_cloudtrail19.json#eventID=0f24b61e-747e-...
6,35e8c621-31e9-4b01-9b09-bb60b4b64638,EVENT,READ,apigateway.amazonaws.com,GetClientCertificates,2020-09-21T22:22:52Z,us-east-1,205.8.181.128,Boto3/1.15.2 Python/3.8.2 Linux/5.6.3-arch1-1 ...,FAILURE,flaws_cloudtrail19.json#eventID=35e8c621-31e9-...
7,992db7f4-8a4b-472a-b983-bd94cab68af7,EVENT,READ,waf.amazonaws.com,GetChangeToken,2020-09-21T22:22:52Z,us-east-1,205.8.181.128,Boto3/1.15.2 Python/3.8.2 Linux/5.6.3-arch1-1 ...,FAILURE,flaws_cloudtrail19.json#eventID=992db7f4-8a4b-...
8,5e4cf454-f463-48e7-a5d8-8e080598d97,EVENT,READ,snowball.amazonaws.com,ListClusters,2020-09-21T22:22:52Z,us-east-1,205.8.181.128,Boto3/1.15.2 Python/3.8.2 Linux/5.6.3-arch1-1 ...,FAILURE,flaws_cloudtrail19.json#eventID=5e4cf454-f463-...
9,233fb77a-f7ae-4607-866a-fa5fb5f48035,EVENT,READ,datapipeline.amazonaws.com,ListPipelines,2020-09-21T22:22:52Z,us-east-1,205.8.181.128,Boto3/1.15.2 Python/3.8.2 Linux/5.6.3-arch1-1 ...,FAILURE,flaws_cloudtrail19.json#eventID=233fb77a-f7ae-...


## 3. `resources.csv`

In [ ]:
# ============================================================
# BLOCK 3 — resources.csv
# ============================================================

resource_store = {}

for record in records:
    event_time = record.get("eventTime") or ""

    for target in extract_targets(record):
        if target["category"] != "RESOURCE":
            continue

        node = dict(target["node"])
        rid = node["resource_id"]

        if rid not in resource_store:
            resource_store[rid] = node
            resource_store[rid]["first_seen"] = event_time
            resource_store[rid]["last_seen"] = event_time
        else:
            if event_time:
                if not resource_store[rid]["first_seen"] or event_time < resource_store[rid]["first_seen"]:
                    resource_store[rid]["first_seen"] = event_time
                if not resource_store[rid]["last_seen"] or event_time > resource_store[rid]["last_seen"]:
                    resource_store[rid]["last_seen"] = event_time

resource_columns = [
    "resource_id",
    "category",
    "resource_type",
    "raw_resource_type",
    "name",
    "arn",
    "service",
    "account_id",
    "region",
]

# 사용자가 확정한 CSV 스키마에는 first_seen/last_seen이 없으므로 export에서는 제외
resources_df = pd.DataFrame(resource_store.values())
if resources_df.empty:
    resources_df = pd.DataFrame(columns=resource_columns)
else:
    resources_df = resources_df[resource_columns].sort_values(
        ["resource_type", "service", "name", "resource_id"]
    ).reset_index(drop=True)

resources_path = OUTPUT_DIR / "resources.csv"
resources_df.to_csv(resources_path, index=False)

print(f"resources.csv: {len(resources_df):,} rows -> {resources_path}")
display(resources_df.head(10))

resources.csv: 22,246 rows -> normalized_csv/resources.csv


,resource_id,category,resource_type,raw_resource_type,name,arn,service,account_id,region
0,aws:resource:a40f0309ac10e3f16162,RESOURCE,COMPUTE,AWS::EC2::Instance,14b6a528b915b3c0270e0174982e5ed78052eafd,,ec2,811596193553,eu-west-1
1,aws:resource:b64b55fff7303d1acabe,RESOURCE,COMPUTE,AWS::EC2::Instance,i-4478871fdbc35f34a,,ec2,811596193553,us-west-2
2,aws:resource:972b46679b24e8dc59fe,RESOURCE,COMPUTE,AWS::EC2::Instance,i-aa2d3b42e5c6e801a,,ec2,811596193553,us-west-2
3,aws:resource:e966a3725993c6c268fe,RESOURCE,COMPUTE,AWS::Lambda::Function,Level6,,lambda,811596193553,us-west-2
4,aws:resource:arn:aws:s3:::accounts-accounts.ap...,RESOURCE,STORAGE,AWS::S3::Bucket,accounts-accounts.apguyver.com,arn:aws:s3:::accounts-accounts.apguyver.com,s3,811596193553,us-west-2
5,aws:resource:arn:aws:s3:::accounts-accounts.ap...,RESOURCE,STORAGE,AWS::S3::Bucket,accounts-accounts.apgyver.com,arn:aws:s3:::accounts-accounts.apgyver.com,s3,811596193553,us-west-2
6,aws:resource:arn:aws:s3:::accounts-accounts.ap...,RESOURCE,STORAGE,AWS::S3::Bucket,accounts-accounts.apparchitect.com,arn:aws:s3:::accounts-accounts.apparchitect.com,s3,811596193553,us-west-2
7,aws:resource:arn:aws:s3:::accounts-accounts.ap...,RESOURCE,STORAGE,AWS::S3::Bucket,accounts-accounts.appguyver.com,arn:aws:s3:::accounts-accounts.appguyver.com,s3,811596193553,us-west-2
8,aws:resource:arn:aws:s3:::accounts-accounts.ap...,RESOURCE,STORAGE,AWS::S3::Bucket,accounts-accounts.appgyver.academy,arn:aws:s3:::accounts-accounts.appgyver.academy,s3,811596193553,us-west-2
9,aws:resource:arn:aws:s3:::accounts-accounts.ap...,RESOURCE,STORAGE,AWS::S3::Bucket,accounts-accounts.appgyver.black,arn:aws:s3:::accounts-accounts.appgyver.black,s3,811596193553,us-west-2


## 4. `relationships.csv`

In [ ]:
# ============================================================
# BLOCK 4 — relationships.csv
# ============================================================

relationship_rows = []
seen_relationships = set()

def add_relationship(source_id, source_category, rel_type,
                     target_id, target_category, event_id,
                     evidence_type, confidence):
    key = (
        source_id, source_category, rel_type,
        target_id, target_category, event_id,
        evidence_type
    )
    if key in seen_relationships:
        return

    seen_relationships.add(key)
    relationship_rows.append({
        "relationship_id": stable_hash(*key, prefix="aws:rel"),
        "source_id": source_id,
        "source_category": source_category,
        "relationship_type": rel_type,
        "target_id": target_id,
        "target_category": target_category,
        "event_id": event_id,
        "evidence_type": evidence_type,
        "confidence": confidence,
    })

for index, record in enumerate(records):
    event_id = record.get("eventID") or stable_hash(
        record.get("eventTime"),
        record.get("eventSource"),
        record.get("eventName"),
        record.get("requestID"),
        index,
        prefix="aws:event",
    )

    # 1) IDENTITY -> EVENT : 누가 이 Event를 수행했는가
    actor = identity_from_user_identity(record)

    add_relationship(
        source_id=actor["identity_id"],
        source_category="IDENTITY",
        rel_type="PERFORMED",
        target_id=event_id,
        target_category="EVENT",
        event_id=event_id,
        evidence_type="DIRECT_CLOUDTRAIL",
        confidence=1.0,
    )

    # 2) AssumedRole Session -> 원래 IAM Role
    issuer = session_issuer_identity(record)
    if issuer and actor["identity_state"] == "SESSION":
        add_relationship(
            source_id=actor["identity_id"],
            source_category="IDENTITY",
            rel_type="SESSION_OF",
            target_id=issuer["identity_id"],
            target_category="IDENTITY",
            event_id=event_id,
            evidence_type="DIRECT_CLOUDTRAIL",
            confidence=1.0,
        )

    # 3) EVENT -> IDENTITY / RESOURCE : Event가 무엇을 대상으로 했는가
    for target in extract_targets(record):
        if target["category"] == "IDENTITY":
            target_id = target["node"]["identity_id"]
        else:
            target_id = target["node"]["resource_id"]

        add_relationship(
            source_id=event_id,
            source_category="EVENT",
            rel_type="TARGETED",
            target_id=target_id,
            target_category=target["category"],
            event_id=event_id,
            evidence_type=target["evidence_type"],
            confidence=target["confidence"],
        )

relationship_columns = [
    "relationship_id",
    "source_id",
    "source_category",
    "relationship_type",
    "target_id",
    "target_category",
    "event_id",
    "evidence_type",
    "confidence",
]

relationships_df = pd.DataFrame(
    relationship_rows,
    columns=relationship_columns
)

relationships_path = OUTPUT_DIR / "relationships.csv"
relationships_df.to_csv(relationships_path, index=False)

# ------------------------------------------------------------
# Referential integrity validation
# ------------------------------------------------------------
identity_ids = set(identities_df["identity_id"])
event_ids = set(events_df["event_id"])
resource_ids = set(resources_df["resource_id"])

valid_ids = {
    "IDENTITY": identity_ids,
    "EVENT": event_ids,
    "RESOURCE": resource_ids,
}

dangling = relationships_df[
    relationships_df.apply(
        lambda row:
            row["source_id"] not in valid_ids.get(row["source_category"], set())
            or row["target_id"] not in valid_ids.get(row["target_category"], set()),
        axis=1
    )
]

print(f"relationships.csv: {len(relationships_df):,} rows -> {relationships_path}")
print(f"Dangling relationships: {len(dangling):,}")
print("\nRelationship type counts:")
display(relationships_df["relationship_type"].value_counts().rename_axis("relationship_type").to_frame("count"))
display(relationships_df.head(10))

relationships.csv: 65,574 rows -> normalized_csv/relationships.csv
Dangling relationships: 0

Relationship type counts:


,count
relationship_type,
PERFORMED,39207
TARGETED,25159
SESSION_OF,1208


,relationship_id,source_id,source_category,relationship_type,target_id,target_category,event_id,evidence_type,confidence
0,aws:rel:1b77a5b87bfb49a5716e,aws:identity:arn:aws:iam::811596193553:user/Le...,IDENTITY,PERFORMED,f146e1b4-5796-40b0-aa15-ba54d7d823d5,EVENT,f146e1b4-5796-40b0-aa15-ba54d7d823d5,DIRECT_CLOUDTRAIL,1.0
1,aws:rel:63b60588f968f13c08b2,aws:identity:arn:aws:iam::811596193553:user/Le...,IDENTITY,PERFORMED,8c58977-02ee-4dc3-8784-7f4bbe90c591,EVENT,8c58977-02ee-4dc3-8784-7f4bbe90c591,DIRECT_CLOUDTRAIL,1.0
2,aws:rel:7887995e677856f581bd,aws:identity:arn:aws:iam::811596193553:user/Le...,IDENTITY,PERFORMED,e2f01b9a-c3d5-4429-9a98-bc7774fb0c41,EVENT,e2f01b9a-c3d5-4429-9a98-bc7774fb0c41,DIRECT_CLOUDTRAIL,1.0
3,aws:rel:10f8bfa0896932d38509,aws:identity:arn:aws:iam::811596193553:user/Le...,IDENTITY,PERFORMED,64220d-0873-4ffb-bb1d-7b9d94c5c773,EVENT,64220d-0873-4ffb-bb1d-7b9d94c5c773,DIRECT_CLOUDTRAIL,1.0
4,aws:rel:6cadc1e65e942a4ab3c0,aws:identity:arn:aws:iam::811596193553:user/Le...,IDENTITY,PERFORMED,d35b87ef-fcbf-4091-802f-a6eafdce7132,EVENT,d35b87ef-fcbf-4091-802f-a6eafdce7132,DIRECT_CLOUDTRAIL,1.0
5,aws:rel:12c7346cc2f9112c5884,aws:identity:arn:aws:iam::811596193553:user/Le...,IDENTITY,PERFORMED,0f24b61e-747e-42f0-ab44-bd9f3cf08c1b,EVENT,0f24b61e-747e-42f0-ab44-bd9f3cf08c1b,DIRECT_CLOUDTRAIL,1.0
6,aws:rel:e395615aa2f2802e7ee2,aws:identity:arn:aws:iam::811596193553:user/Le...,IDENTITY,PERFORMED,35e8c621-31e9-4b01-9b09-bb60b4b64638,EVENT,35e8c621-31e9-4b01-9b09-bb60b4b64638,DIRECT_CLOUDTRAIL,1.0
7,aws:rel:9d338d5fd862906daaf1,aws:identity:arn:aws:iam::811596193553:user/Le...,IDENTITY,PERFORMED,992db7f4-8a4b-472a-b983-bd94cab68af7,EVENT,992db7f4-8a4b-472a-b983-bd94cab68af7,DIRECT_CLOUDTRAIL,1.0
8,aws:rel:9ea383c6017c8c6318be,aws:identity:arn:aws:iam::811596193553:user/Le...,IDENTITY,PERFORMED,5e4cf454-f463-48e7-a5d8-8e080598d97,EVENT,5e4cf454-f463-48e7-a5d8-8e080598d97,DIRECT_CLOUDTRAIL,1.0
9,aws:rel:3c109e30e8454fef4b7a,aws:identity:arn:aws:iam::811596193553:user/Le...,IDENTITY,PERFORMED,233fb77a-f7ae-4607-866a-fa5fb5f48035,EVENT,233fb77a-f7ae-4607-866a-fa5fb5f48035,DIRECT_CLOUDTRAIL,1.0
